In [ ]:
# Instalar dependencias necesarias
!pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib openpyxl pandas

import io
import os
import hashlib
import pandas as pd
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload

# Autenticación con Google Drive
auth.authenticate_user()
drive_service = build('drive', 'v3')

print("Autenticación completada con éxito.")

In [ ]:
# CONFIGURACIÓN: Reemplaza estos valores con los IDs de tus carpetas
ID_CARPETA_ORIGEN = 'TU_ID_DE_CARPETA_ORIGEN_AQUI'
ID_CARPETA_DESTINO = 'TU_ID_DE_CARPETA_DESTINO_AQUI'

# Nombre de la columna anonimizada resultante
NOMBRE_COLUMNA_ANONIMIZADA = 'ID_Anonimizado'

def generar_id_unico(valor):
    """
    Convierte un valor de carnet/carné en un ID único corto y seguro usando SHA-256.
    Ejemplo: '12345678' -> 'ID-a1b2c3d4'
    """
    if pd.isna(valor) or str(valor).strip() == '':
        return None
    val_str = str(valor).strip().encode('utf-8')
    hash_hex = hashlib.sha256(val_str).hexdigest()[:8] # Primeros 8 caracteres
    return f"ID-{hash_hex}"

def listar_archivos_excel(folder_id):
    """Obtiene la lista de archivos Excel dentro de la carpeta origen."""
    query = f"'{folder_id}' in parents and trashed = false and (mimeType = 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet' or mimeType = 'application/vnd.ms-excel')"
    results = drive_service.files().list(q=query, fields="files(id, name)").execute()
    return results.get('files', [])

def procesar_archivos():
    archivos = listar_archivos_excel(ID_CARPETA_ORIGEN)

    if not archivos:
        print("No se encontraron archivos de Excel en la carpeta de origen.")
        return

    print(f"Se encontraron {len(archivos)} archivos. Iniciando procesamiento...\n")

    for archivo in archivos:
        file_id = archivo['id']
        file_name = archivo['name']
        print(f"Procesando: {file_name}...")

        # 1. Descargar el archivo a memoria
        request = drive_service.files().get_media(fileId=file_id)
        fh = io.BytesIO()
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()
        fh.seek(0)

        try:
            # 2. Leer únicamente la primera hoja
            excel_obj = pd.ExcelFile(fh, engine='openpyxl')
            primera_hoja = excel_obj.sheet_names[0]
            df = pd.read_excel(excel_obj, sheet_name=primera_hoja)

            # 3. Eliminar columnas solicitadas si existen
            columnas_a_borrar = ["Teléfono", "Documento", "Apellidos y Nombres"]
            columnas_existentes = [col for col in columnas_a_borrar if col in df.columns]
            if columnas_existentes:
                df.drop(columns=columnas_existentes, inplace=True)
                print(f"  - Columnas eliminadas: {columnas_existentes}")

            # 4. Anonimizar columna "Carnet" o "Carné"
            col_carnet = None
            if "Carnet" in df.columns:
                col_carnet = "Carnet"
            elif "Carné" in df.columns:
                col_carnet = "Carné"

            if col_carnet:
                df[col_carnet] = df[col_carnet].apply(generar_id_unico)
                df.rename(columns={col_carnet: NOMBRE_COLUMNA_ANONIMIZADA}, inplace=True)
                print(f"  - Columna '{col_carnet}' anonimizada a '{NOMBRE_COLUMNA_ANONIMIZADA}'.")

            # 5. Guardar la hoja modificada en un nuevo archivo temporal
            nuevo_nombre_archivo = f"Procesado_{file_name}"
            ruta_temporal = f"/tmp/{nuevo_nombre_archivo}"

            with pd.ExcelWriter(ruta_temporal, engine='openpyxl') as writer:
                df.to_excel(writer, index=False, sheet_name=primera_hoja)

            # 6. Subir el archivo resultante a la carpeta destino en Google Drive
            file_metadata = {
                'name': nuevo_nombre_archivo,
                'parents': [ID_CARPETA_DESTINO]
            }
            media = MediaFileUpload(ruta_temporal, mimetype='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet')
            drive_service.files().create(body=file_metadata, media_body=media, fields='id').execute()

            # Limpiar archivo temporal
            if os.path.exists(ruta_temporal):
                os.remove(ruta_temporal)

            print(f"  ✓ Guardado con éxito en Drive: {nuevo_nombre_archivo}\n")

        except Exception as e:
            print(f"  ✕ Error al procesar {file_name}: {str(e)}\n")

    print("¡Proceso completado para todos los archivos!")

# Ejecutar la función
procesar_archivos()